In [ ]:
# Clone the repository to bring in the database and files
# !git clone https://github.com/amirreza1998/lora-medical-finetuning.git
# %cd lora-medical-finetuning

# Install Colab-specific requirements
!pip install -q -r requirements_colab.txt

In [ ]:
# Login to Weights & Biases for experiment tracking
# Paste your API key when prompted (get it from https://wandb.ai/authorize)
import wandb
wandb.login(key="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")

/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: amirrezakazemloo99 (amirrezakazemloo99-k-n-toosi-university-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [ ]:
# pyrefly: ignore [missing-import]
import torch
import json
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import (
    prepare_model_for_kbit_training,
    LoraConfig,
    get_peft_model,
    PeftModel,
    PeftConfig
)
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
import gc
import os
from dotenv import load_dotenv

In [ ]:
# Improved configuration
# --- ABLATION STUDY PARAMETERS ---
EXPERIMENT_NAME = 'qwen2.5_lora_r32_neftune_on' # CHANGE THIS FOR EACH RUN!

class Config:
    load_dotenv()
    # No HF Token needed for Qwen 2.5!
    MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct" # Or "Qwen/Qwen2.5-14B-Instruct" if you use A100
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    MAX_LENGTH = 1024 # Increased slightly to accommodate rich medical context
    OUTPUT_DIR = f'./results_{EXPERIMENT_NAME}'
    FINAL_MODEL_DIR = f'./final_model_{EXPERIMENT_NAME}'

def load_base_model():
    """Enhanced model loading with better error handling for Qwen/Llama architectures"""
    print(f"Loading base model {Config.MODEL_NAME}...")

    try:
        tokenizer = AutoTokenizer.from_pretrained(
            Config.MODEL_NAME,
            trust_remote_code=True,
            padding_side="right" # Right padding is recommended for causal LM
        )

        # Better padding configuration
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        # Optimized quantization config for Colab
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_storage=torch.uint8
        )

        model = AutoModelForCausalLM.from_pretrained(
            Config.MODEL_NAME,
            quantization_config=bnb_config,
            device_map="auto", # Automatically maps to A100/L4 GPU
            trust_remote_code=True,
            torch_dtype=torch.bfloat16,
            low_cpu_mem_usage=True
        )

        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
        return model, tokenizer

    except Exception as e:
        print(f"Error loading model: {e}")
        raise

In [ ]:
def prepare_training_data_enhanced(data_path: str, tokenizer, validation_split: float = 0.15):
    """Data preparation utilizing tokenizer.apply_chat_template for dynamic prompt construction"""

    print("Preparing training data...")

    with open(data_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    utterances = data.get("Utterance", [])
    formatted_data = []

    for item in utterances:
        query = item.get("query", "")
        diseases = item.get("diseases", [])

        if not query or not diseases:
            continue

        diseases_text = "، ".join(diseases)

        # Create conversational format compatible with apply_chat_template
        messages = [
            {"role": "system", "content": "شما یک دستیار پزشکی متخصص هستید که بر اساس علائم ارائه شده، بیماری‌های محتمل را تشخیص می‌دهید. پاسخ خود را به صورت دقیق و علمی ارائه دهید."},
            {"role": "user", "content": query},
            {"role": "assistant", "content": f"با توجه به علائم ذکر شده، بیماری‌های محتمل عبارتند از: {diseases_text}\n\nبرای تشخیص دقیق‌تر، توصیه می‌شود با پزشک متخصص مشورت کنید و آزمایش‌های لازم را انجام دهید."}
        ]

        # This will format the text exactly how the specific model (e.g. Qwen2.5) expects it!
        formatted_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)
        formatted_data.append({"text": formatted_text})

    df = pd.DataFrame(formatted_data)
    dataset = Dataset.from_pandas(df)

    # Notice we don't manually tokenize and set labels=input_ids anymore.
    # TRL's SFTTrainer handles the tokenization and label masking for us!

    split_dataset = dataset.train_test_split(
        test_size=validation_split,
        seed=42,
        shuffle=True
    )

    return split_dataset

In [ ]:
def create_optimized_lora_config():
    """Optimized LoRA config using DoRA for better performance."""
    return LoraConfig(
        r=32,  # Increased rank for better capacity
        lora_alpha=64,
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ],
        use_dora=True # Use Weight-Decomposed Low-Rank Adaptation (DoRA)
    )

def train_model_enhanced(model, tokenizer, dataset, lora_config):
    """Enhanced training with TRL's SFTTrainer, response masking, and NEFTune"""
    print("Clearing GPU memory...")
    torch.cuda.empty_cache()
    gc.collect()

    training_args = TrainingArguments(
        output_dir=Config.OUTPUT_DIR,
        num_train_epochs=1,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        gradient_checkpointing=True,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_steps=100,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=3,
        fp16=False,
        bf16=True, # bfloat16 is better and fully supported on A100/L4
        dataloader_pin_memory=False,
        remove_unused_columns=True, # MUST be True so string columns are removed before collation
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=25,
        optim="paged_adamw_8bit", # Changed to 8-bit optimizer to save VRAM
        max_grad_norm=1.0,
        weight_decay=0.05,
        neftune_noise_alpha=5.0 # NEFTune noise helps reduce overfitting and improves generation
    )

    # We use DataCollatorForCompletionOnlyLM to ensure the model ONLY calculates loss on the assistant's response.
    # It finds the tokens corresponding to the start of the assistant's generation and sets prior tokens to -100.
    response_template = "<|im_start|>assistant\n"
    collator = DataCollatorForCompletionOnlyLM(response_template, tokenizer=tokenizer)

    print("Starting SFTTrainer...")
    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["test"],
        peft_config=lora_config,
        dataset_text_field="text",
        max_seq_length=Config.MAX_LENGTH,
        tokenizer=tokenizer,
        data_collator=collator, #this part make magic masking
        packing=False
    )

    try:
        training_result = trainer.train()
        print(f"Training completed. Final loss: {training_result.training_loss:.4f}")

        trainer.model.save_pretrained(Config.FINAL_MODEL_DIR, safe_serialization=True)
        tokenizer.save_pretrained(Config.FINAL_MODEL_DIR)

        return trainer.model

    except Exception as e:
        print(f"Error during training: {e}")
        raise

In [ ]:
def interactive_test(model_path: str):
    """Interactive testing interface"""
    print("Loading model for interactive testing...")

    config = PeftConfig.from_pretrained(model_path)
    tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path, trust_remote_code=True)

    base_model = AutoModelForCausalLM.from_pretrained(
        config.base_model_name_or_path,
        load_in_4bit=True,
        device_map="auto"
    )

    model = PeftModel.from_pretrained(base_model, model_path)
    model.eval()

    print("\n🩺 Medical assistant is ready! (Type 'exit' to quit)")

    while True:
        user_input = input("\nDescribe your symptoms (or type 'exit' to quit): ").strip()
        if user_input.lower() in ['exit', 'خروج', 'quit']: break
        if not user_input: continue

        messages = [
            {"role": "system", "content": "شما یک دستیار پزشکی متخصص هستید که بر اساس علائم ارائه شده، بیماری‌های محتمل را تشخیص می‌دهید."},
            {"role": "user", "content": user_input}
        ]

        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=256,
                temperature=0.3,
                do_sample=True,
                top_p=0.85,
                repetition_penalty=1.15,
                pad_token_id=tokenizer.eos_token_id
            )

        # Get only the generated part
        generated_ids = outputs[0][inputs['input_ids'].shape[-1]:]
        response = tokenizer.decode(generated_ids, skip_special_tokens=True)
        print(f"\n🩺 Medical Assistant response:\n{response}")

def main():
    print("🚀 Starting Advanced Medical SFT model training")

    # Change this path to match your Colab structure
    dataset_path = "./database/intent_instruction_datasets/multi_intent_disease_queries(method3).json"

    if not os.path.exists(dataset_path):
        print(f"❌ Dataset file not found at {dataset_path}. Please upload the file and correct the path.")
        return

    model, tokenizer = load_base_model()
    dataset = prepare_training_data_enhanced(dataset_path, tokenizer)
    lora_config = create_optimized_lora_config()

    trained_model = train_model_enhanced(model, tokenizer, dataset, lora_config)
    print("✅ Training completed!")

    print("\n🎯 Starting interactive testing...")
    interactive_test(Config.FINAL_MODEL_DIR)

if __name__ == "__main__":
    main()

🚀 Starting Advanced Medical SFT model training
Loading base model Qwen/Qwen2.5-7B-Instruct...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Training completed. Final loss: 0.2076
✅ Training completed!

🎯 Starting interactive testing...
Loading model for interactive testing...


The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


🩺 Medical assistant is ready! (Type 'exit' to quit)

Describe your symptoms (or type 'exit' to quit): hi


KeyboardInterrupt: 

In [ ]:
!du -h /content/lora-medical-finetuning

12K	/content/lora-medical-finetuning/results_qwen2.5_lora_r32_neftune_on/runs/Oct09_13-23-08_9ffbb56a8c85
16K	/content/lora-medical-finetuning/results_qwen2.5_lora_r32_neftune_on/runs
334M	/content/lora-medical-finetuning/results_qwen2.5_lora_r32_neftune_on/checkpoint-130
334M	/content/lora-medical-finetuning/results_qwen2.5_lora_r32_neftune_on
344K	/content/lora-medical-finetuning/database/disease_knowledge_base
1.2M	/content/lora-medical-finetuning/database/intent_instruction_datasets
1.5M	/content/lora-medical-finetuning/database
172M	/content/lora-medical-finetuning/final_model_qwen2.5_lora_r32_neftune_on
12K	/content/lora-medical-finetuning/medical_lora_output/runs/Oct09_08-17-26_30ad58918ee7
12K	/content/lora-medical-finetuning/medical_lora_output/runs/Oct09_08-10-23_30ad58918ee7
12K	/content/lora-medical-finetuning/medical_lora_output/runs/Oct09_07-47-10_30ad58918ee7
40K	/content/lora-medical-finetuning/medical_lora_output/runs
44K	/content/lora-medical-finetuning/medical_lora_o

In [ ]:
!cp -r /content/lora-medical-finetuning /content/drive/MyDrive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Automatically install required evaluation libraries
try:
    import evaluate
except ImportError:
    !pip install -q evaluate rouge_score
    import evaluate

import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

def evaluate_and_plot(trainer, tokenizer, dataset, experiment_name, num_samples=50):
    print(f"Evaluating {experiment_name} on {num_samples} test samples...")
    rouge = evaluate.load('rouge')

    # Random test subset
    test_data = dataset['test'].shuffle(seed=42).select(range(min(num_samples, len(dataset['test']))))

    predictions = []
    references = []

    model = trainer.model
    model.eval()

    for item in tqdm(test_data):
        text = item['text']
        prompt_end = text.find("<|im_start|>assistant\n") + len("<|im_start|>assistant\n")
        prompt = text[:prompt_end]
        actual_response = text[prompt_end:].replace("<|im_end|>", "").strip()

        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                temperature=0.1,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )

        generated_ids = outputs[0][inputs['input_ids'].shape[-1]:]
        generated_response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

        predictions.append(generated_response)
        references.append(actual_response)

    # Calculate ROUGE scores
    results = rouge.compute(predictions=predictions, references=references)

    # Plot Evaluation
    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(8, 5))
    metrics = ['rouge1', 'rouge2', 'rougeL']
    scores = [results[m] * 100 for m in metrics]
    ax = sns.barplot(x=metrics, y=scores, palette="viridis")
    plt.title(f'ROUGE Scores: {experiment_name}', fontsize=16, fontweight='bold')
    plt.ylabel('Score (%)', fontsize=12)
    plt.ylim(0, 100)
    for i, v in enumerate(scores):
        ax.text(i, v + 1.5, f"{v:.1f}%", ha='center', fontweight='bold', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{experiment_name}_rouge_metrics.pdf', dpi=300)
    plt.show()

    # Plot Training Loss History
    plt.figure(figsize=(10, 6))
    history = trainer.state.log_history
    train_steps = [x['step'] for x in history if 'loss' in x]
    train_loss = [x['loss'] for x in history if 'loss' in x]
    eval_steps = [x['step'] for x in history if 'eval_loss' in x]
    eval_loss = [x['eval_loss'] for x in history if 'eval_loss' in x]

    plt.plot(train_steps, train_loss, label='Training Loss', color='blue', alpha=0.7, linewidth=2)
    if eval_loss:
        plt.plot(eval_steps, eval_loss, label='Validation Loss', color='red', marker='o', linewidth=2)
    plt.title(f'Loss Curve: {experiment_name}', fontsize=16, fontweight='bold')
    plt.xlabel('Training Steps', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.legend(fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{experiment_name}_loss_curve.pdf', dpi=300)
    plt.show()

    # Save Qualitative Results
    df_results = pd.DataFrame({
        "Ground Truth": references,
        "Model Prediction": predictions
    })
    df_results.to_csv(f"{experiment_name}_qualitative.csv", index=False, encoding="utf-8-sig")

    print(f"✅ Ablation study files saved for {experiment_name}:")
    print(f"  - {experiment_name}_rouge_metrics.pdf")
    print(f"  - {experiment_name}_loss_curve.pdf")
    print(f"  - {experiment_name}_qualitative.csv")

    return results

# if __name__ == "__main__":
# Ensure this runs only if trainer is available (so we don't crash the nb logic if someone just runs all cells blindly)
# try:
if 'trainer' in globals() and 'EXPERIMENT_NAME' in globals():
    evaluate_and_plot(trainer, tokenizer, dataset, EXPERIMENT_NAME)
# except Exception as e:
#     pass

### Force GPU Execution for Model and Evaluation

In [ ]:
import torch
import gc
from peft import PeftModel, PeftConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. Clear GPU memory completely first
torch.cuda.empty_cache()
gc.collect()

# 2. Re-load the model but explicitly force CUDA placement
base_dir = "/content/lora-medical-finetuning"
model_path = f"{base_dir}/final_model_qwen2.5_lora_r32_neftune_on"

print("Loading config & tokenizer...")
config = PeftConfig.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(config.base_model_name_or_path, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print("Loading Base Model in 4-bit with strict GPU mapping...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

# Force device_map to use device 0 instead of "auto" to avoid any CPU offloading
gpu_device_map = {"": 0}

base_model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    quantization_config=bnb_config,
    device_map=gpu_device_map,
    trust_remote_code=True,
    torch_dtype=torch.bfloat16
)

print("Loading LoRA adapters onto GPU...")
model_peft = PeftModel.from_pretrained(base_model, model_path, device_map=gpu_device_map)
model_peft.eval()

# Recreate mock trainer pointing to the strict GPU model
trainer = MockTrainer(model_peft)
print("\n✅ Success! Model loaded entirely on GPU. Now try running the test_single_generation cell above.")

Loading config & tokenizer...
Loading Base Model in 4-bit with strict GPU mapping...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Loading LoRA adapters onto GPU...

✅ Success! Model loaded entirely on GPU. Now try running the test_single_generation cell above.


### Test Generation on a Single Sample with Verbose Memory Tracking

In [ ]:
import torch
import time

def test_single_generation(model, tokenizer, dataset):
    # Print current GPU Memory allocation
    if torch.cuda.is_available():
        print(f"Initial GPU Allocated Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
        print(f"Initial GPU Cached Memory: {torch.cuda.memory_reserved() / 1e9:.2f} GB")

    # Pick 1 sample
    sample = dataset['test'][0]
    text = sample['text']
    prompt_end = text.find("<|im_start|>assistant\n") + len("<|im_start|>assistant\n")
    prompt = text[:prompt_end]

    print("\n--- Prompt ---")
    print(prompt)
    print("---------------\n")

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    print("Generating response (timing started)... ")
    start_time = time.time()

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64, # reduced to speed up testing
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    end_time = time.time()
    print(f"Generation completed in {end_time - start_time:.2f} seconds!")

    generated_ids = outputs[0][inputs['input_ids'].shape[-1]:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    print("\n--- Model Output ---")
    print(response)
    print("--------------------")

test_single_generation(trainer.model, tokenizer, dataset)

Initial GPU Allocated Memory: 5.95 GB
Initial GPU Cached Memory: 6.55 GB

--- Prompt ---
<|im_start|>system
شما یک دستیار پزشکی متخصص هستید که بر اساس علائم ارائه شده، بیماری‌های محتمل را تشخیص می‌دهید. پاسخ خود را به صورت دقیق و علمی ارائه دهید.<|im_end|>
<|im_start|>user
حرکات غیرارادی تو پاهام دارم و نمی‌تونم درست بخوابم، در کنارش قلبم تند می‌زنه و سرگیجه دارم.<|im_end|>
<|im_start|>assistant

---------------

Generating response (timing started)... 
Generation completed in 97.85 seconds!

--- Model Output ---
با توجه به علائم ذکر شده، بیماری‌های محتمل عبارتند از: سندرم پای بی‌قرار، پرفشاری خون، دیسک کمر

برای تشخیص دقیق‌
--------------------


### plot

In [ ]:
# Automatically install required evaluation libraries
try:
    import evaluate
except ImportError:
    !pip install -q evaluate rouge_score
    import evaluate

import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

def evaluate_and_plot(trainer, tokenizer, dataset, experiment_name, num_samples=50):
    print(f"Evaluating {experiment_name} on {num_samples} test samples...")
    rouge = evaluate.load('rouge')

    # Random test subset
    test_data = dataset['test'].shuffle(seed=42).select(range(min(num_samples, len(dataset['test']))))

    predictions = []
    references = []

    model = trainer.model
    model.eval()

    for item in tqdm(test_data):
        text = item['text']
        import pdb;pdb.set_trace()
        prompt_end = text.find("<|im_start|>assistant\n") + len("<|im_start|>assistant\n")
        prompt = text[:prompt_end]
        actual_response = text[prompt_end:].replace("<|im_end|>", "").strip()

        inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=128,
                temperature=0.1,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )

        generated_ids = outputs[0][inputs['input_ids'].shape[-1]:]
        generated_response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

        predictions.append(generated_response)
        references.append(actual_response)

    # Calculate ROUGE scores
    results = rouge.compute(predictions=predictions, references=references)

    # Plot Evaluation
    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(8, 5))
    metrics = ['rouge1', 'rouge2', 'rougeL']
    scores = [results[m] * 100 for m in metrics]
    ax = sns.barplot(x=metrics, y=scores, palette="viridis")
    plt.title(f'ROUGE Scores: {experiment_name}', fontsize=16, fontweight='bold')
    plt.ylabel('Score (%)', fontsize=12)
    plt.ylim(0, 100)
    for i, v in enumerate(scores):
        ax.text(i, v + 1.5, f"{v:.1f}%", ha='center', fontweight='bold', fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{experiment_name}_rouge_metrics.pdf', dpi=300)
    plt.show()

    # Plot Training Loss History
    plt.figure(figsize=(10, 6))
    history = trainer.state.log_history
    train_steps = [x['step'] for x in history if 'loss' in x]
    train_loss = [x['loss'] for x in history if 'loss' in x]
    eval_steps = [x['step'] for x in history if 'eval_loss' in x]
    eval_loss = [x['eval_loss'] for x in history if 'eval_loss' in x]

    plt.plot(train_steps, train_loss, label='Training Loss', color='blue', alpha=0.7, linewidth=2)
    if eval_loss:
        plt.plot(eval_steps, eval_loss, label='Validation Loss', color='red', marker='o', linewidth=2)
    plt.title(f'Loss Curve: {experiment_name}', fontsize=16, fontweight='bold')
    plt.xlabel('Training Steps', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.legend(fontsize=12)
    plt.tight_layout()
    plt.savefig(f'{experiment_name}_loss_curve.pdf', dpi=300)
    plt.show()

    # Save Qualitative Results
    df_results = pd.DataFrame({
        "Ground Truth": references,
        "Model Prediction": predictions
    })
    df_results.to_csv(f"{experiment_name}_qualitative.csv", index=False, encoding="utf-8-sig")

    print(f"✅ Ablation study files saved for {experiment_name}:")
    print(f"  - {experiment_name}_rouge_metrics.pdf")
    print(f"  - {experiment_name}_loss_curve.pdf")
    print(f"  - {experiment_name}_qualitative.csv")

    return results

if 'trainer' in globals() and 'EXPERIMENT_NAME' in globals():
    print("\n--- Starting Generation & Evaluation ---")
    # Running with 5 samples for speed (approx 5-7 minutes). Change to 50 for full eval.
    final_results = evaluate_and_plot(trainer, tokenizer, dataset, EXPERIMENT_NAME, num_samples=5)
    print("\nFinal ROUGE Scores:", final_results)


--- Starting Generation & Evaluation ---
Evaluating qwen2.5_lora_r32_neftune_on on 5 test samples...


  0%|          | 0/5 [00:00<?, ?it/s]

> /tmp/ipykernel_994/3966951981.py(29)evaluate_and_plot()
     27     for item in tqdm(test_data):
     28         text = item['text']
---> 29         import pdb;pdb.set_trace()
     30         prompt_end = text.find("<|im_start|>assistant\n") + len("<|im_start|>assistant\n")
     31         prompt = text[:prompt_end]

ipdb> text
'<|im_start|>system\nشما یک دستیار پزشکی متخصص هستید که بر اساس علائم ارائه شده، بیماری\u200cهای محتمل را تشخیص می\u200cدهید. پاسخ خود را به صورت دقیق و علمی ارائه دهید.<|im_end|>\n<|im_start|>user\nیه برآمدگی تو شکمم حس می\u200cکنم و با سرفه بزرگ\u200cتر می\u200cشه، در کنارش سرفه\u200cهای شدید دارم و نفس کشیدن برام سخت شده.<|im_end|>\n<|im_start|>assistant\nبا توجه به علائم ذکر شده، بیماری\u200cهای محتمل عبارتند از: فتق، بیماری مزمن انسدادی ریه، دیورتیکولیت\n\nبرای تشخیص دقیق\u200cتر، توصیه می\u200cشود با پزشک متخصص مشورت کنید و آزمایش\u200cهای لازم را انجام دهید.<|im_end|>\n'
ipdb> n
> /tmp/ipykernel_994/3966951981.py(30)evaluate_and_plot()
     28         t